# Unit III: Text as Data

## Is climate change an endangered-species threat? Thirty years of the Endangered Species Act

In 2019, [Delach et al.](https://doi.org/10.1038/s41558-019-0620-8) asked whether the agencies that
implement the U.S. Endangered Species Act take climate change seriously. They read the official
documents for all 459 endangered animals by hand. Climate change was named as a threat for 64% of
them, and management actions to address it were planned for 18%. Their per-species codings are
[openly published](https://doi.org/10.17605/OSF.IO/R9UCA).

In this module we read a related body of documents with code, and compare what we find with what
they found by hand.

## The legal record

Under section 4 of the ESA, a species is listed as endangered or threatened because of one or more
of five factors:

- **(A)** the present or threatened destruction, modification, or curtailment of its habitat or range
- **(B)** overutilization for commercial, recreational, scientific, or educational purposes
- **(C)** disease or predation
- **(D)** the inadequacy of existing regulatory mechanisms
- **(E)** other natural or manmade factors affecting its continued existence

The decision must be made "solely on the basis of the best scientific and commercial data
available." Each final listing decision is published as a rule in the
[Federal Register](https://www.federalregister.gov/). The rule lays out the evidence, responds to
public and peer-review comments, and states the agency's conclusion. These rules are legal documents
that can be challenged in court, and they are written with that in mind.

## The question

When the U.S. government lists a species under the ESA, does it say climate change is a reason?
How has that changed since the 1990s, and how does your answer compare with Delach et al.?

How you make this question precise is up to you, and it is the main thing you will be graded on.

## Data

- **Federal Register API** (no key needed): <https://www.federalregister.gov/developers/documentation/api/v1>.
  Fish and Wildlife Service rules are titled "Endangered and Threatened Wildlife and Plants; …", and
  each document's full text is available from its `raw_text_url`.
- **Delach et al. (2019) per-species data:** <https://osf.io/download/76ytr/>, with column
  definitions at <https://osf.io/download/4bm8u/>.

## Tools

This module introduces five ways to work with text. You will need to decide which to use for each
step, and you should be able to say why.

- **Regular expressions:** exact patterns over text. They are fast, transparent and literal.
- **[Laya](https://huggingface.co/convaiinnovations/laya):** an open-weights *decision model*. You
  give it a text and typed questions, and it returns a probability for each answer instead of
  generating text. A `noul` question returns the probability of "yes", a `choice` question returns
  a probability for each label you define, and a `score` question places the text on a scale you
  describe. It runs on a GPU on the National Research Platform.
- **Embeddings:** each passage becomes a vector of 4096 numbers, placed so that passages with
  similar meaning are close together. Stored in DuckDB, they can be searched by meaning with
  `array_cosine_similarity`.
- **Retrieval-augmented generation (RAG):** retrieve the passages most relevant to a question, then
  ask a language model to answer from those passages only, citing them.
- **Structured extraction:** a language model fills in a schema you define (for example a
  [Pydantic](https://docs.pydantic.dev/) model), one record per document.

The language model is `deepseek-v4-flash` and the embedding model is `qwen3-embedding`, both served
by the [National Research Platform](https://nrp.ai/documentation/userdocs/ai/llm-managed/) and
called here through [LangChain](https://python.langchain.com/).

## Setup

This cell connects to the three services. It reads your `NRP_TOKEN` and `LAYA_API_KEY` from the
environment.

In [ ]:
import os
import requests
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

nrp = dict(base_url="https://ellm.nrp-nautilus.io/v1", api_key=os.environ["NRP_TOKEN"])
llm = ChatOpenAI(model="deepseek-v4-flash", temperature=0, timeout=180,
                 extra_body={"thinking": {"type": "disabled"}}, **nrp)
embedder = OpenAIEmbeddings(model="qwen3-embedding", check_embedding_ctx_length=False, **nrp)

laya_url = "https://laya.carlboettiger.info"
laya_auth = {"Authorization": f"Bearer {os.environ['LAYA_API_KEY']}"}

def decide(text, questions):
    response = requests.post(f"{laya_url}/v1/systemone", headers=laya_auth,
                             json={"state": text, "questions": questions})
    response.raise_for_status()
    return response.json()["answers"]

In [ ]:
decide("Streamflow will decline as the climate warms, drying the creeks where the species breeds.",
       {"climate_threat": {"type": "noul", "instructions": "Does this text say climate change threatens the species?"}})

Your session has about 4 GB of memory. Forty million characters of rule text fit comfortably, but
some ways of reshaping them do not.